# Phase 3 — Chunk-Size Experiments

This notebook runs the controlled 250, 500, 750, and 1000-token chunk-size comparison while holding overlap at 75 tokens. It delegates to the tested runner and summarizes the generated comparison artifact.

Running the suite rebuilds only guarded `phase3-*` Pinecone namespaces.

## Setup

Use the repository `.venv` kernel. The runner loads `.env` with override enabled.

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import yaml
from IPython.display import JSON, display

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in (Path.cwd(), Path.cwd().parent)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")


def run_command(command: list[object]) -> None:
    normalized = [str(part) for part in command]
    print(shlex.join(normalized))
    subprocess.run(normalized, cwd=PROJECT_ROOT, check=True)


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_yaml(path: Path) -> dict:
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}


print(f"Project root: {PROJECT_ROOT}")
print(f"Kernel Python: {sys.executable}")


## Configuration

Review the controlled YAML matrix before opting into the external run.

In [ ]:
CONFIG_DIR = PROJECT_ROOT / "config" / "experiments"
QUESTIONS_PATH = PROJECT_ROOT / "evaluation" / "questions.json"
OUTPUT_ROOT = PROJECT_ROOT / "evaluation" / "results" / "phase3_chunking"
RUN_EXPERIMENT = False  # Set to True, then run the execution cell.

CONFIG_FILES = sorted(CONFIG_DIR.glob("*.yaml"))
print(f"Found {len(CONFIG_FILES)} configurations:")
for config_path in CONFIG_FILES:
    print(f"- {config_path.relative_to(PROJECT_ROOT)}")
    display(JSON(read_yaml(config_path)))

## Run the controlled suite

In [ ]:
COMMAND = [
    sys.executable,
    PROJECT_ROOT / "evaluation" / "run_chunk_experiments.py",
    "--configs", CONFIG_DIR,
    "--questions", QUESTIONS_PATH,
    "--output-root", OUTPUT_ROOT,
]

if RUN_EXPERIMENT:
    run_command(COMMAND)
else:
    print("Dry run. Set RUN_EXPERIMENT = True to execute:")
    print(shlex.join(str(part) for part in COMMAND))

## Compare results

In [ ]:
COMPARISON_PATH = OUTPUT_ROOT / "comparison.json"
SUMMARY_FIELDS = ["experiment_id","chunk_size","chunk_overlap","chunk_count","recall_at_5","retrieval_failure_count","mean_total_latency_seconds"]

if COMPARISON_PATH.exists():
    comparison = read_json(COMPARISON_PATH)
    rows = [
        {field: experiment.get(field) for field in SUMMARY_FIELDS}
        for experiment in comparison.get("experiments", [])
    ]
    display(JSON({
        "phase": comparison.get("phase"),
        "best_recall_at_5": comparison.get("best_recall_at_5"),
        "best_recall_experiment_ids": comparison.get("best_recall_experiment_ids"),
        "experiments": rows,
    }))
else:
    print(f"No comparison artifact yet: {COMPARISON_PATH}")

## Deeper inspection

Open an experiment's `results.json` under the output directory to inspect per-question retrieval ranks, answers, citations, failures, and latency.